# Step 2: Feature Engineering - Spatial Homologation
In this step, we use geospatial mathematics (cKDTree and 3D spherical projection) to match every Emergency Room center to its geographically closest environmental monitoring station (SINCA) and Climatology station (DMC).

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import math
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time
import re

# Project Rule 3: Text output must be generated for all visuals/mappings
print("=== SPATIAL HOMOLOGATION: Hospitals to SINCA & DMC Stations ===")

# 1. Load Hospitals
df_hosp = pd.read_csv('../data/raw/Atenciones de Urgencia/Establecimientos de Salud/establecimientos_20260714.csv', sep=';', encoding='latin1')
df_hosp = df_hosp.dropna(subset=['Latitud', 'Longitud'])
# Clean coordinates
df_hosp['Latitud'] = df_hosp['Latitud'].astype(str).str.replace(',', '.').astype(float)
df_hosp['Longitud'] = df_hosp['Longitud'].astype(str).str.replace(',', '.').astype(float)

hospitals_coords = df_hosp[['EstablecimientoCodigoAntiguo', 'EstablecimientoGlosa', 'Latitud', 'Longitud']].drop_duplicates(subset=['EstablecimientoCodigoAntiguo']).copy()

# 2. Load SINCA Stations
sinca_csv_path = '../data/raw/Registros de Calidad del Aire/Reportes SINCA/Data/DatosEstacioneSINCA.csv'
df_sinca = pd.read_csv(sinca_csv_path, sep=';', decimal='.', encoding='utf-8', skiprows=1)
df_sinca = df_sinca.dropna(subset=['latitud', 'longitud'])
sinca_coords = df_sinca[['estacion', 'region', 'latitud', 'longitud']].drop_duplicates(subset=['estacion']).reset_index(drop=True).copy()

print(f"Loaded {len(hospitals_coords)} Hospitals with valid coordinates.")
print(f"Loaded {len(sinca_coords)} SINCA Stations with valid coordinates.")

# 3. Convert Lat/Lon to 3D Cartesian coordinates for accurate spherical distance using cKDTree (SINCA)
R = 6371.0 # Earth radius in kilometers
sinca_coords['x'] = R * np.cos(np.radians(sinca_coords['latitud'])) * np.cos(np.radians(sinca_coords['longitud']))
sinca_coords['y'] = R * np.cos(np.radians(sinca_coords['latitud'])) * np.sin(np.radians(sinca_coords['longitud']))
sinca_coords['z'] = R * np.sin(np.radians(sinca_coords['latitud']))

hospitals_coords['x'] = R * np.cos(np.radians(hospitals_coords['Latitud'])) * np.cos(np.radians(hospitals_coords['Longitud']))
hospitals_coords['y'] = R * np.cos(np.radians(hospitals_coords['Latitud'])) * np.sin(np.radians(hospitals_coords['Longitud']))
hospitals_coords['z'] = R * np.sin(np.radians(hospitals_coords['Latitud']))

# Build the KDTree & Query (SINCA)
tree_sinca = cKDTree(sinca_coords[['x', 'y', 'z']])
distances_sinca, indices_sinca = tree_sinca.query(hospitals_coords[['x', 'y', 'z']])

hospitals_coords['nearest_sinca_station'] = sinca_coords.loc[indices_sinca, 'estacion'].values
hospitals_coords['nearest_sinca_region'] = sinca_coords.loc[indices_sinca, 'region'].values
hospitals_coords['distance_to_sinca_km'] = distances_sinca 

print("\n--- Textual Data Output for Spatial Homologation (SINCA) (Top 15) ---")
display_df = hospitals_coords[['EstablecimientoGlosa', 'nearest_sinca_station', 'distance_to_sinca_km']].head(15)
print(display_df.to_string(index=False))
print("----------------------------------------------------------------------")

# 4. Load & Geocode DMC Climatology Stations
print("\n--- Processing DMC Climatology Stations ---")
dmc_path = '../data/raw/Datos Climatológicos/Anuario Climatológico/processed/dataset_climatologico_final_perfecto.csv'
try:
    df_dmc = pd.read_csv(dmc_path, encoding='utf-8')
except UnicodeDecodeError:
    df_dmc = pd.read_csv(dmc_path, encoding='latin1')

unique_dmc = df_dmc['Estacion_Normalizada'].unique()
unique_dmc = [s for s in unique_dmc if str(s).upper() not in ['ESTACION NO DETECTADA', 'NAN', 'NONE']]

dmc_coords_path = '../data/processed/dmc_stations_coords.csv'
if os.path.exists(dmc_coords_path):
    dmc_coords = pd.read_csv(dmc_coords_path)
    print(f"Loaded existing DMC coordinates for {len(dmc_coords)} stations.")
else:
    print(f"Geocoding {len(unique_dmc)} unique DMC stations (this will take a couple of minutes)...")
    geolocator = Nominatim(user_agent="vitalflow_chile_health_project", timeout=10)
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)
    
    records = []
    for station in unique_dmc:
        # Clean the station name safely using regex
        clean_name = str(station).upper()
        clean_name = re.sub(r'[^A-Z\s,]', '', clean_name)
        clean_name = clean_name.replace(' AP', '').replace(' AD', '').replace(' CMA', '').strip()
        
        if "EDUARDO FREI" in clean_name: query = "Base Eduardo Frei, Antartica, Chile"
        elif "JUAN F ERN ANDE Z" in clean_name: query = "Archipielago Juan Fernandez, Chile"
        elif "HAITE N" in clean_name: query = "Chaiten, Chile"
        else: query = f"{clean_name}, Chile"
        
        try:
            location = geocode(query)
            if location and location.latitude != -33.4377: # Exclude generic "Chile" match (Santiago center)
                records.append({
                    'Estacion_Normalizada': station,
                    'latitud_dmc': location.latitude,
                    'longitud_dmc': location.longitude
                })
            else:
                if "," in query:
                    fallback_query = query.split(",")[1].strip() + ", Chile"
                    loc2 = geocode(fallback_query)
                    if loc2 and loc2.latitude != -33.4377:
                        records.append({
                            'Estacion_Normalizada': station,
                            'latitud_dmc': loc2.latitude,
                            'longitud_dmc': loc2.longitude
                        })
                    else:
                        print(f"  Could not geocode: {station} -> {query}")
                else:
                    print(f"  Could not geocode: {station} -> {query}")
        except Exception as e:
            print(f"  Error geocoding {station}: {e}")
            
    dmc_coords = pd.DataFrame(records)
    dmc_coords.to_csv(dmc_coords_path, index=False)
    print(f"Geocoding completed. Saved to {dmc_coords_path}")

# 5. Spatial Homologation (Hospitals to DMC)
dmc_coords = dmc_coords.dropna(subset=['latitud_dmc', 'longitud_dmc'])
dmc_coords = dmc_coords.reset_index(drop=True)

if len(dmc_coords) > 0:
    dmc_coords['x'] = R * np.cos(np.radians(dmc_coords['latitud_dmc'])) * np.cos(np.radians(dmc_coords['longitud_dmc']))
    dmc_coords['y'] = R * np.cos(np.radians(dmc_coords['latitud_dmc'])) * np.sin(np.radians(dmc_coords['longitud_dmc']))
    dmc_coords['z'] = R * np.sin(np.radians(dmc_coords['latitud_dmc']))

    tree_dmc = cKDTree(dmc_coords[['x', 'y', 'z']])
    distances_dmc, indices_dmc = tree_dmc.query(hospitals_coords[['x', 'y', 'z']])

    hospitals_coords['nearest_dmc_station'] = dmc_coords.loc[indices_dmc, 'Estacion_Normalizada'].values
    hospitals_coords['distance_to_dmc_km'] = distances_dmc

    print("\n--- Textual Data Output for Spatial Homologation (DMC) (Top 15 mappings) ---")
    display_dmc = hospitals_coords[['EstablecimientoGlosa', 'nearest_dmc_station', 'distance_to_dmc_km']].head(15)
    print(display_dmc.to_string(index=False))
    print("----------------------------------------------------------------------")
else:
    print("\nNo DMC stations could be geocoded.")

# Overwrite mapping with both SINCA and DMC
out_path = '../data/processed/hospital_sinca_dmc_mapping.csv'
hospitals_coords.to_csv(out_path, index=False)
print(f"\nFinal combined mapping successfully saved to {out_path}")

=== SPATIAL HOMOLOGATION: Hospitals to SINCA & DMC Stations ===
Loaded 4025 Hospitals with valid coordinates.
Loaded 199 SINCA Stations with valid coordinates.

--- Textual Data Output for Spatial Homologation (SINCA) (Top 15) ---
                                           EstablecimientoGlosa nearest_sinca_station  distance_to_sinca_km
                                 Servicio MÃ©dico Legal Iquique         Alto Hospicio              8.688653
                                   Laboratorio ClÃ­nico Austral            Trapén Sur            120.641117
                                                   SUR Dalcahue            Trapén Sur            106.440711
                                  Posta de Salud Rural Cascadas          Puerto Varas             39.267417
                     Centro de RehabilitaciÃ³n de MinusvÃ¡lidos                Osorno             36.964984
                                   ClÃ­nica del Trabajador AChS              La Union              1.234795
             

Geocoding 164 unique DMC stations (this will take a couple of minutes)...


  Could not geocode: CARRIEL SUR, CONCEPCIÓN AP. -> CARRIEL SUR, CONCEPCIN, Chile


  Could not geocode: GENERAL FREIRE, CURICÓ AD., -> GENERAL FREIRE, CURIC,, Chile


  Could not geocode: GENERAL BERNARDO O'HIGGINS, CHILLÁN AD. -> GENERAL BERNARDO OHIGGINS, CHILLN, Chile


  Could not geocode: MARÍA DOLORES, LOS ÁNGELES AD. -> MARA DOLORES, LOS NGELES, Chile


Geocoding completed. Saved to ../data/processed/dmc_stations_coords.csv

--- Textual Data Output for Spatial Homologation (DMC) (Top 15 mappings) ---
                                           EstablecimientoGlosa                     nearest_dmc_station  distance_to_dmc_km
                                 Servicio MÃ©dico Legal Iquique             IQUIQUE - DIEGO ARACENA AP.           36.865228
                                   Laboratorio ClÃ­nico Austral                            QUELLON AD .           71.978984
                                                   SUR Dalcahue                            QUELLON AD .           82.414597
                                  Posta de Salud Rural Cascadas              EL TEPUAL PUERTO MONTT AP.           53.374864
                     Centro de RehabilitaciÃ³n de MinusvÃ¡lidos C.M.A. EDUARDO FREI MONTALVA, ANTÁRTICA           35.733548
                                   ClÃ­nica del Trabajador AChS                  CAÑAL BAJO, OSORNO AD.   